# sim_dual_arm — Google Colab Pro ACT 训练流程（rclone 版）

本 Notebook 通过 **VS Code Colab 插件**（或网页端 Colab）连接云端 GPU 执行 **ACT 模型训练**，全程使用 **`rclone`** 对接 Google Drive（默认 `gdrive:sim_dual_arm`），**无需 `drive.mount` 重复弹窗登录**。

## 前置步骤（在本机执行）

1. 在本机 `learning` 容器内完成数据转换：
   ```bash
   ./env.sh exec learning scripts/convert.sh --robot <robot> --task <task>
   ```
2. 使用配套的本机同步 Notebook [sync_drive_rclone.ipynb](sync_drive_rclone.ipynb)（选择本机 Python 内核），修改 `ROBOT` 与 `TASK` 后运行**第 3 节**将 `lerobot.tar` 上传至 `gdrive:sim_dual_arm/<robot>/<task>/datasets/`。

## 使用步骤

1. 在 VS Code 右上角点击 **Select Kernel** → **Colab** → 连接 GPU 运行时（L4 / A100 或高 RAM T4）。
2. 在下方 **第 1 节** 按需修改 `ROBOT`、`TASK`、`MAX_STEPS` 与视觉增强覆盖参数（`VISUAL_AUGMENT` / `CAMERA_DROPOUT_PROB`）。
3. 按顺序运行全部 Cell：**第 2 节**通过 `rclone` 连接 `gdrive:sim_dual_arm`（首次新运行时可通过环境变量、Colab Secrets 或遮罩输入框注入 `rclone.conf`）；**第 6 节**实测最优训练参数；**第 7 节**执行训练并每 30 秒打印实时进度；**第 8 节**用 `rclone` 将模型和训练日志回传 Google Drive。
4. 训练完成后，在本机打开 [sync_drive_rclone.ipynb](sync_drive_rclone.ipynb) 运行**第 5 节**即可一键拉回模型与日志。


In [ ]:
# 1. 参数配置（按需修改后运行）
import os

REPO_URL = "https://github.com/ljk4/sim_dual_arm.git"
REPO_BRANCH = "main"
WORK_DIR = "/content/sim_dual_arm"
RCLONE_REMOTE = "gdrive:sim_dual_arm"

# 按需切换任务：如 marvin_sim / stack 或 marvin_sim / handover
ROBOT = "marvin_sim"
TASK = "handover"
TRAIN_MODE = "full"   # 可选: smoke / overfit / full
MAX_STEPS = 20000     # None 使用配方默认步数；可设整数如 5000 / 20000
DATASET_ARCHIVE = "lerobot.tar"
USE_DRIVE_CACHE = True
TORCH_INDEX_URL = ""  # 留空自动按显卡驱动选择 cu128 / cu126

# 视觉增强开关覆盖（None 表示沿用配方默认值：enabled=True, camera_dropout_prob=0.10）
VISUAL_AUGMENT = None       # 可设为 True / False / None
CAMERA_DROPOUT_PROB = None  # 可设为 0.0 ~ 1.0 浮点数或 None

# 服务器资源最大化探测与覆盖选项
AUTO_PROBE = True         # True 时在第 6 节实测最优 batch_size / num_workers / prefetch_factor
MAX_BATCH_SIZE = 256      # 探测 batch_size 上限（受限于 TrainingConfig <= 256）
VRAM_TARGET_RATIO = 0.85  # 允许占用的最大显存比例
RAM_TARGET_RATIO = 0.70   # 允许占用的最大可用内存 / /dev/shm 比例
BATCH_SIZE = None         # 若显式填写整数，将覆盖探测/配方值
NUM_WORKERS = None        # 若显式填写整数，将覆盖探测/配方值
PREFETCH_FACTOR = None    # 若显式填写整数，将覆盖探测/配方值
SCALE_LR_SQRT = False     # True 时按 sqrt(batch_size / base_batch_size) 缩放 lr / lr_backbone

os.environ.update({
    "SDA_REPO_URL": REPO_URL,
    "SDA_REPO_BRANCH": REPO_BRANCH,
    "SDA_WORK_DIR": WORK_DIR,
    "SDA_RCLONE_REMOTE": RCLONE_REMOTE,
    "SDA_ROBOT": ROBOT,
    "SDA_TASK": TASK,
    "SDA_TRAIN_MODE": TRAIN_MODE,
    "SDA_MAX_STEPS": "" if MAX_STEPS is None else str(int(MAX_STEPS)),
    "SDA_DATASET_ARCHIVE": DATASET_ARCHIVE,
    "SDA_USE_DRIVE_CACHE": "1" if USE_DRIVE_CACHE else "0",
    "SDA_TORCH_INDEX_URL": TORCH_INDEX_URL,
    "SDA_VISUAL_AUGMENT": "" if VISUAL_AUGMENT is None else ("1" if VISUAL_AUGMENT else "0"),
    "SDA_CAMERA_DROPOUT_PROB": "" if CAMERA_DROPOUT_PROB is None else str(float(CAMERA_DROPOUT_PROB)),
    "SDA_AUTO_PROBE": "1" if AUTO_PROBE else "0",
    "SDA_MAX_BATCH_SIZE": str(int(MAX_BATCH_SIZE)),
    "SDA_VRAM_TARGET_RATIO": str(float(VRAM_TARGET_RATIO)),
    "SDA_RAM_TARGET_RATIO": str(float(RAM_TARGET_RATIO)),
    "SDA_BATCH_SIZE": "" if BATCH_SIZE is None else str(int(BATCH_SIZE)),
    "SDA_NUM_WORKERS": "" if NUM_WORKERS is None else str(int(NUM_WORKERS)),
    "SDA_PREFETCH_FACTOR": "" if PREFETCH_FACTOR is None else str(int(PREFETCH_FACTOR)),
    "SDA_SCALE_LR_SQRT": "1" if SCALE_LR_SQRT else "0",
})
print(
    f"配置已就绪: {ROBOT}/{TASK} | mode={TRAIN_MODE} | max_steps={MAX_STEPS} | "
    f"remote={RCLONE_REMOTE} | visual_aug={VISUAL_AUGMENT} | cam_drop={CAMERA_DROPOUT_PROB} | "
    f"auto_probe={AUTO_PROBE} | max_bs={MAX_BATCH_SIZE} | scale_lr_sqrt={SCALE_LR_SQRT}"
)

In [ ]:
# 2. 检查 GPU 并配置 rclone 连接 Google Drive（免 drive.mount 登录）
import base64
import getpass
import os
import shutil
import subprocess
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True)
if smi.stdout:
    print(smi.stdout)
if smi.stderr:
    print(smi.stderr)

if shutil.which("rclone") is None:
    print("正在安装 rclone...")
    rc = subprocess.run(
        ["bash", "-lc", "curl -fsSL https://rclone.org/install.sh | bash"],
        check=False,
    )
    if rc.returncode != 0 or shutil.which("rclone") is None:
        subprocess.run(
            ["bash", "-lc", "apt-get update -qq && apt-get install -y -qq rclone"],
            check=True,
        )

remote_name = RCLONE_REMOTE.split(":", 1)[0]
conf_path = Path.home() / ".config" / "rclone" / "rclone.conf"
conf_path.parent.mkdir(parents=True, exist_ok=True)

def _has_remote() -> bool:
    if not conf_path.is_file():
        return False
    res = subprocess.run(["rclone", "listremotes"], check=False, capture_output=True, text=True)
    return f"{remote_name}:" in res.stdout.splitlines()

def _decode_rclone_conf(raw_text: str) -> str:
    text = raw_text.strip()
    if text.startswith("[") and f"[{remote_name}]" in text:
        return text + "\n"
    decoded = base64.b64decode(text).decode("utf-8")
    if f"[{remote_name}]" not in decoded:
        raise ValueError(f"提供的 rclone 配置中未找到 [{remote_name}] 节")
    return decoded.rstrip() + "\n"

if _has_remote():
    print(f"已检测到现有 rclone remote '{remote_name}:' ({conf_path})，直接复用。")
else:
    raw_conf = os.environ.get("RCLONE_CONF_B64") or os.environ.get("RCLONE_CONF") or ""
    if not raw_conf:
        try:
            from google.colab import userdata
            raw_conf = userdata.get("RCLONE_CONF_B64") or userdata.get("RCLONE_CONF") or ""
        except Exception:
            raw_conf = ""
    if not raw_conf:
        print(
            "未检测到 rclone.conf。请在本机终端运行 `base64 -w0 ~/.config/rclone/rclone.conf` "
            "并粘贴到下方遮罩输入框（支持 base64 或原始配置文本，不会明文回显）："
        )
        raw_conf = getpass.getpass("RCLONE_CONF_B64: ")
    conf_path.write_text(_decode_rclone_conf(raw_conf), encoding="utf-8")
    os.chmod(conf_path, 0o600)

lsd = subprocess.run(["rclone", "lsd", RCLONE_REMOTE], check=False, capture_output=True, text=True)
if lsd.returncode != 0:
    raise RuntimeError(f"rclone 连接 {RCLONE_REMOTE} 失败:\n{lsd.stderr}")

for sub in (
    f"{RCLONE_REMOTE}/_cache",
    f"{RCLONE_REMOTE}/{ROBOT}/{TASK}/datasets",
    f"{RCLONE_REMOTE}/{ROBOT}/{TASK}/models",
    f"{RCLONE_REMOTE}/{ROBOT}/{TASK}/logs/training",
):
    subprocess.run(["rclone", "mkdir", sub], check=True)

print(f"rclone 已就绪并连通: {RCLONE_REMOTE}/{ROBOT}/{TASK}")

In [ ]:
%%bash
# 3. 克隆或更新公开仓库到 Colab 本地盘
set -euo pipefail

if [[ -d "$SDA_WORK_DIR/.git" ]]; then
  echo "仓库已存在，更新分支 $SDA_REPO_BRANCH..."
  git -C "$SDA_WORK_DIR" fetch origin "$SDA_REPO_BRANCH"
  git -C "$SDA_WORK_DIR" checkout "$SDA_REPO_BRANCH"
  git -C "$SDA_WORK_DIR" pull --ff-only origin "$SDA_REPO_BRANCH"
else
  echo "克隆公开仓库 $SDA_REPO_URL ($SDA_REPO_BRANCH) 到 $SDA_WORK_DIR..."
  git clone --branch "$SDA_REPO_BRANCH" --single-branch "$SDA_REPO_URL" "$SDA_WORK_DIR"
fi

echo -n "当前提交: "
git -C "$SDA_WORK_DIR" log -1 --oneline

In [ ]:
%%bash
# 4. 安装学习端环境与第三方源码/权重（通过 rclone 复用 _cache）
set -euo pipefail
cd "$SDA_WORK_DIR"

CACHE_LOCAL="/tmp/sda_rclone_cache"
args=("--repo-root" "$SDA_WORK_DIR")
if [[ "$SDA_USE_DRIVE_CACHE" == "1" ]]; then
  mkdir -p "$CACHE_LOCAL"
  echo "从 $SDA_RCLONE_REMOTE/_cache 拉取缓存..."
  rclone copy "$SDA_RCLONE_REMOTE/_cache" "$CACHE_LOCAL" --transfers 4 || true
  args+=("--cache-dir" "$CACHE_LOCAL")
fi
if [[ -n "$SDA_TORCH_INDEX_URL" ]]; then
  args+=("--torch-index-url" "$SDA_TORCH_INDEX_URL")
fi

bash docs/colab/setup_colab_learning.sh "${args[@]}"

if [[ "$SDA_USE_DRIVE_CACHE" == "1" ]]; then
  echo "回传新增缓存到 $SDA_RCLONE_REMOTE/_cache..."
  rclone copy "$CACHE_LOCAL" "$SDA_RCLONE_REMOTE/_cache" --transfers 4 || true
fi

In [ ]:
# 5. 通过 rclone 从 Google Drive 拉取数据集到 Colab 本地盘
import json
import shutil
import subprocess
from pathlib import Path

remote_datasets = f"{RCLONE_REMOTE}/{ROBOT}/{TASK}/datasets"
local_datasets = Path(WORK_DIR) / "var" / ROBOT / TASK / "datasets"
local_datasets.mkdir(parents=True, exist_ok=True)

target_dir = local_datasets / "lerobot"
staging_dir = local_datasets / ".unpack_tmp"
if staging_dir.exists():
    shutil.rmtree(staging_dir)
staging_dir.mkdir(parents=True, exist_ok=True)

tmp_archive = Path("/tmp") / f"{ROBOT}_{TASK}_{DATASET_ARCHIVE}"

def _remote_files(remote_dir: str) -> set[str]:
    res = subprocess.run(
        ["rclone", "lsf", remote_dir, "--files-only"],
        check=False,
        capture_output=True,
        text=True,
    )
    if res.returncode != 0:
        return set()
    return {line.strip() for line in res.stdout.splitlines() if line.strip()}

try:
    top_files = _remote_files(remote_datasets)
    if DATASET_ARCHIVE in top_files:
        remote_archive = f"{remote_datasets}/{DATASET_ARCHIVE}"
        print(f"正在通过 rclone 下载压缩包: {remote_archive} -> {tmp_archive}")
        subprocess.run(["rclone", "copyto", remote_archive, str(tmp_archive), "--progress"], check=True)
        print(f"正在解压到本地盘: {staging_dir}")
        subprocess.run(["tar", "-xf", str(tmp_archive), "-C", str(staging_dir)], check=True)
        if (staging_dir / "lerobot" / "manifest.json").is_file():
            extracted_root = staging_dir / "lerobot"
        elif (staging_dir / "manifest.json").is_file():
            extracted_root = staging_dir
        else:
            raise FileNotFoundError(
                f"{remote_archive} 解压后未找到 manifest.json（应在根级或 lerobot/ 下）"
            )
    elif "manifest.json" in _remote_files(f"{remote_datasets}/lerobot"):
        remote_unpacked = f"{remote_datasets}/lerobot"
        print(f"未找到 {DATASET_ARCHIVE}，改用远端已解压目录: {remote_unpacked}")
        extracted_root = staging_dir / "lerobot"
        subprocess.run(
            ["rclone", "copy", remote_unpacked, str(extracted_root), "--transfers", "8", "--progress"],
            check=True,
        )
    else:
        raise FileNotFoundError(
            f"在 {remote_datasets} 未找到 {DATASET_ARCHIVE} 或 lerobot/manifest.json，"
            "请先在本机运行 docs/colab/sync_drive_rclone.ipynb 上传数据集"
        )

    if target_dir.exists():
        shutil.rmtree(target_dir)
    shutil.move(str(extracted_root), str(target_dir))
finally:
    if tmp_archive.exists():
        tmp_archive.unlink()
    if staging_dir.exists():
        shutil.rmtree(staging_dir, ignore_errors=True)

required = ("manifest.json", "train", "validation", "processors")
missing = [name for name in required if not (target_dir / name).exists()]
if missing:
    raise RuntimeError(f"本地数据集目录缺少必要项: {missing} ({target_dir})")

manifest = json.loads((target_dir / "manifest.json").read_text(encoding="utf-8"))
splits = manifest.get("splits", {})
tr = splits.get("training", {})
va = splits.get("validation", {})
print(f"本地数据集就绪: {target_dir}")
print(
    f"status={manifest.get('status')} | synthetic={manifest.get('synthetic')} | "
    f"train={tr.get('episode_count')} 条/{tr.get('frame_count')} 帧 | "
    f"validation={va.get('episode_count')} 条/{va.get('frame_count')} 帧"
)

In [ ]:
# 6. 探测当前服务器最优训练参数（最大化利用 GPU 显存与 CPU/RAM）
import json
import shlex
import subprocess
from pathlib import Path

work = Path(WORK_DIR)
probe_output = work / "var" / ROBOT / TASK / "debug" / "colab_probe_result.json"

if not AUTO_PROBE:
    print("AUTO_PROBE=False，跳过硬件吞吐探测。")
else:
    probe_args = [
        "python3",
        "docs/colab/probe_colab_training.py",
        "--robot",
        ROBOT,
        "--task",
        TASK,
        "--mode",
        TRAIN_MODE,
        "--max-batch-size",
        str(int(MAX_BATCH_SIZE)),
        "--vram-target",
        str(float(VRAM_TARGET_RATIO)),
        "--ram-target",
        str(float(RAM_TARGET_RATIO)),
        "--output",
        str(probe_output),
    ]
    cmd_str = "cd " + shlex.quote(str(work)) + " && source .env.learning && exec " + shlex.join(probe_args)
    proc = subprocess.Popen(
        ["bash", "-lc", cmd_str],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
    ret = proc.wait()
    if ret != 0:
        raise subprocess.CalledProcessError(ret, probe_args)
    report = json.loads(probe_output.read_text(encoding="utf-8"))
    print("已保存探测推荐参数:", json.dumps(report["recommended"], ensure_ascii=False))

In [ ]:
# 7. 启动 ACT 训练（自动应用最优参数与视觉增强，每 30 秒汇报实时进度）
import json
import math
import os
import shlex
import subprocess
import threading
from pathlib import Path

work = Path(WORK_DIR)
venv_python = work / ".venv" / "bin" / "python"
src_cfg = work / "shared" / "scenarios" / ROBOT / TASK / "configs" / "learning" / "act_training.yaml"
probe_output = work / "var" / ROBOT / TASK / "debug" / "colab_probe_result.json"

def _fmt_hms(seconds):
    if seconds is None:
        return "--:--:--"
    total = max(0, round(seconds))
    hours, rem = divmod(total, 3600)
    minutes, secs = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{secs:02d}"

base_info = json.loads(
    subprocess.run(
        [
            str(venv_python),
            "-c",
            (
                "import json, sys, yaml; "
                "doc = yaml.safe_load(open(sys.argv[1], encoding='utf-8')); "
                "tr = doc['training']; "
                "va = tr.get('visual_augmentation') or {}; "
                "print(json.dumps({"
                "'amp_dtype': tr.get('amp_dtype', 'bfloat16'), "
                "'batch_size': int(tr['batch_size']), "
                "'num_workers': int(tr.get('num_workers', 4)), "
                "'prefetch_factor': int(tr.get('prefetch_factor', 2)), "
                "'lr': float(tr['lr']), "
                "'lr_backbone': None if tr.get('lr_backbone') is None else float(tr['lr_backbone']), "
                "'visual_aug_enabled': bool(va.get('enabled', False)), "
                "'camera_dropout_prob': float(va.get('camera_dropout_prob', 0.10)), "
                "'default_steps': int(doc['modes'][sys.argv[2]]['steps'])"
                "}))"
            ),
            str(src_cfg),
            TRAIN_MODE,
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

base_amp_dtype = base_info["amp_dtype"]
base_batch_size = base_info["batch_size"]
base_num_workers = base_info["num_workers"]
base_prefetch_factor = base_info["prefetch_factor"]
base_lr = base_info["lr"]
base_lr_backbone = base_info["lr_backbone"]
base_aug_enabled = base_info["visual_aug_enabled"]
base_cam_dropout = base_info["camera_dropout_prob"]
default_steps = base_info["default_steps"]

if AUTO_PROBE and probe_output.is_file():
    probe_report = json.loads(probe_output.read_text(encoding="utf-8"))
    rec = probe_report["recommended"]
    target_amp_dtype = str(rec["amp_dtype"])
    target_batch_size = int(rec["batch_size"])
    target_num_workers = int(rec["num_workers"])
    target_prefetch_factor = int(rec["prefetch_factor"])
    param_source = f"硬件探测 ({probe_output.name})"
else:
    gpu_probe = json.loads(
        subprocess.run(
            [
                str(venv_python),
                "-c",
                (
                    "import json, torch; "
                    "cuda_ok = torch.cuda.is_available(); "
                    "cap = list(torch.cuda.get_device_capability(0)) if cuda_ok else [0, 0]; "
                    "print(json.dumps({'amp_dtype': 'bfloat16' if (cuda_ok and cap[0] >= 8) else 'float16'}))"
                ),
            ],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip()
    )
    target_amp_dtype = str(gpu_probe["amp_dtype"])
    target_batch_size = base_batch_size
    target_num_workers = min(8, max(2, os.cpu_count() or 2))
    target_prefetch_factor = base_prefetch_factor
    param_source = "默认配方 + 硬件能力检测"

if BATCH_SIZE is not None:
    target_batch_size = int(BATCH_SIZE)
if NUM_WORKERS is not None:
    target_num_workers = int(NUM_WORKERS)
if PREFETCH_FACTOR is not None:
    target_prefetch_factor = int(PREFETCH_FACTOR)

target_lr = base_lr
target_lr_backbone = base_lr_backbone
if SCALE_LR_SQRT and target_batch_size != base_batch_size:
    lr_scale = math.sqrt(target_batch_size / base_batch_size)
    target_lr = base_lr * lr_scale
    if base_lr_backbone is not None:
        target_lr_backbone = base_lr_backbone * lr_scale

target_aug_enabled = base_aug_enabled if VISUAL_AUGMENT is None else bool(VISUAL_AUGMENT)
target_cam_dropout = base_cam_dropout if CAMERA_DROPOUT_PROB is None else float(CAMERA_DROPOUT_PROB)
target_steps = int(MAX_STEPS) if MAX_STEPS is not None else default_steps
exceeds_default_steps = MAX_STEPS is not None and int(MAX_STEPS) > default_steps

needs_temp_cfg = (
    target_amp_dtype != base_amp_dtype
    or target_batch_size != base_batch_size
    or target_num_workers != base_num_workers
    or target_prefetch_factor != base_prefetch_factor
    or target_lr != base_lr
    or target_lr_backbone != base_lr_backbone
    or target_aug_enabled != base_aug_enabled
    or target_cam_dropout != base_cam_dropout
    or exceeds_default_steps
)

print(f"参数来源: {param_source}")
print(
    f"有效训练参数: batch_size={target_batch_size} | num_workers={target_num_workers} | "
    f"prefetch_factor={target_prefetch_factor} | amp_dtype={target_amp_dtype} | "
    f"lr={target_lr:g} | lr_backbone={target_lr_backbone} | steps={target_steps} | "
    f"visual_aug={target_aug_enabled} | camera_dropout_prob={target_cam_dropout:g}"
)

train_args = ["bash", "learning/lerobot/scripts/train.sh"]
if needs_temp_cfg:
    debug_dir = work / "var" / ROBOT / TASK / "debug"
    debug_dir.mkdir(parents=True, exist_ok=True)
    temp_cfg = debug_dir / "act_training_colab.yaml"
    overrides = {
        "amp_dtype": target_amp_dtype,
        "batch_size": target_batch_size,
        "num_workers": target_num_workers,
        "prefetch_factor": target_prefetch_factor,
        "lr": target_lr,
        "lr_backbone": target_lr_backbone,
        "visual_aug_enabled": target_aug_enabled,
        "camera_dropout_prob": target_cam_dropout,
        "steps": int(MAX_STEPS) if exceeds_default_steps else None,
    }
    subprocess.run(
        [
            str(venv_python),
            "-c",
            (
                "import json, sys, yaml; "
                "src, dst, mode, raw = sys.argv[1:5]; "
                "ov = json.loads(raw); "
                "doc = yaml.safe_load(open(src, encoding='utf-8')); "
                "doc['training']['amp_dtype'] = ov['amp_dtype']; "
                "doc['training']['batch_size'] = ov['batch_size']; "
                "doc['training']['num_workers'] = ov['num_workers']; "
                "doc['training']['prefetch_factor'] = ov['prefetch_factor']; "
                "doc['training']['lr'] = ov['lr']; "
                "doc['training']['lr_backbone'] = ov['lr_backbone']; "
                "va = dict(doc['training'].get('visual_augmentation') or {}); "
                "va['enabled'] = ov['visual_aug_enabled']; "
                "va['camera_dropout_prob'] = ov['camera_dropout_prob']; "
                "doc['training']['visual_augmentation'] = va; "
                "(doc['modes'][mode].update({'steps': ov['steps']}) if ov['steps'] is not None else None); "
                "yaml.safe_dump(doc, open(dst, 'w', encoding='utf-8'), sort_keys=False, allow_unicode=True)"
            ),
            str(src_cfg),
            str(temp_cfg),
            TRAIN_MODE,
            json.dumps(overrides),
        ],
        check=True,
    )
    print(f"已写入临时训练配置: {temp_cfg}")
    train_args.extend(["--config", str(temp_cfg)])
else:
    print("使用方案默认训练配方。")
    train_args.extend(["--robot", ROBOT, "--task", TASK])

train_args.extend(["--mode", TRAIN_MODE])
if MAX_STEPS is not None:
    train_args.extend(["--max-steps", str(int(MAX_STEPS))])

logs_root = work / "var" / ROBOT / TASK / "logs" / "training"
existing_runs = (
    {p.name for p in logs_root.iterdir() if p.is_dir() and p.name.startswith("act_")}
    if logs_root.is_dir()
    else set()
)

stop_event = threading.Event()
print_lock = threading.Lock()

def _watch_progress() -> None:
    active_metrics = None
    last_train_step = 0
    last_val_step = -1
    best_val = None
    while not stop_event.wait(30.0):
        if active_metrics is None and logs_root.is_dir():
            new_runs = sorted(
                (
                    p
                    for p in logs_root.iterdir()
                    if p.is_dir() and p.name.startswith("act_") and p.name not in existing_runs
                ),
                key=lambda p: p.name,
            )
            if new_runs:
                candidate = new_runs[-1] / "metrics.jsonl"
                if candidate.is_file():
                    active_metrics = candidate
        if active_metrics is None or not active_metrics.is_file():
            continue
        try:
            lines = active_metrics.read_text(encoding="utf-8").splitlines()
        except OSError:
            continue
        latest_train = None
        latest_val = None
        for raw in lines:
            if not raw.strip():
                continue
            try:
                row = json.loads(raw)
            except json.JSONDecodeError:
                continue
            phase = row.get("phase")
            if phase == "train":
                latest_train = row
            elif phase == "validation":
                latest_val = row
                val_l1 = float(row["val_l1"])
                if best_val is None or val_l1 < best_val:
                    best_val = val_l1
        if latest_val is not None:
            val_step = int(latest_val["step"])
            if val_step > last_val_step:
                last_val_step = val_step
                with print_lock:
                    print(
                        f"[验证] step={val_step} | val_l1={float(latest_val['val_l1']):.5f} | best_val={best_val:.5f}",
                        flush=True,
                    )
        if latest_train is not None:
            step = int(latest_train["step"])
            if step > last_train_step:
                last_train_step = step
                elapsed = float(latest_train.get("elapsed_seconds", 0.0))
                pct = min(100.0, step / max(1, target_steps) * 100.0)
                eta = (elapsed * max(0, target_steps - step) / step) if step > 0 else None
                best_part = f" | best_val={best_val:.5f}" if best_val is not None else ""
                with print_lock:
                    print(
                        f"[进度] {step}/{target_steps} ({pct:5.1f}%) | "
                        f"loss={float(latest_train['loss']):.5f} | "
                        f"l1={float(latest_train.get('l1_loss', 0.0)):.5f} | "
                        f"grad={float(latest_train.get('grad_norm', 0.0)):.2f}{best_part} | "
                        f"用时 {_fmt_hms(elapsed)} | 预计剩余 {_fmt_hms(eta)}",
                        flush=True,
                    )

cmd_str = "cd " + shlex.quote(str(work)) + " && source .env.learning && exec " + shlex.join(train_args)
with print_lock:
    print("执行命令:", shlex.join(train_args), flush=True)

watcher = threading.Thread(target=_watch_progress, name="colab-train-progress", daemon=True)
watcher.start()
proc = subprocess.Popen(
    ["bash", "-lc", cmd_str],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
try:
    assert proc.stdout is not None
    for line in proc.stdout:
        with print_lock:
            print(line, end="", flush=True)
    ret = proc.wait()
finally:
    stop_event.set()
    watcher.join(timeout=2.0)
if ret != 0:
    raise subprocess.CalledProcessError(ret, train_args)

In [ ]:
# 8. 通过 rclone 同步最新模型与训练日志回 Google Drive
import subprocess
from pathlib import Path

local_var = Path(WORK_DIR) / "var" / ROBOT / TASK
models_dir = local_var / "models"
candidates = sorted(
    (p for p in models_dir.iterdir() if p.is_dir() and p.name.startswith("act_")),
    key=lambda p: p.name,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError(f"未找到训练产出的模型目录: {models_dir}")

latest_model = candidates[0]
if not (latest_model / "model.safetensors").is_file():
    raise FileNotFoundError(f"模型目录缺少 model.safetensors: {latest_model}")

remote_task_root = f"{RCLONE_REMOTE}/{ROBOT}/{TASK}"
remote_model = f"{remote_task_root}/models/{latest_model.name}"
subprocess.run(
    ["rclone", "copy", str(latest_model), remote_model, "--transfers", "4", "--progress"],
    check=True,
)

src_log = local_var / "logs" / "training" / latest_model.name
remote_log = f"{remote_task_root}/logs/training/{latest_model.name}"
if src_log.is_dir():
    subprocess.run(
        ["rclone", "copy", str(src_log), remote_log, "--transfers", "4", "--progress"],
        check=True,
    )

debug_dir = local_var / "debug"
for extra_name in ("colab_probe_result.json", "act_training_colab.yaml"):
    extra_file = debug_dir / extra_name
    if extra_file.is_file():
        subprocess.run(
            ["rclone", "copyto", str(extra_file), f"{remote_log}/{extra_name}"],
            check=True,
        )

print(f"模型已同步至 Google Drive: {remote_model}")
for item in sorted(latest_model.iterdir()):
    if item.is_file():
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f"  - {item.name}: {size_mb:.2f} MiB")
print(f"训练日志与探测记录已同步至 Google Drive: {remote_log}")
print("\n在本机运行 docs/colab/sync_drive_rclone.ipynb 第 5 节即可一键下载模型至:")
print(f"  var/{ROBOT}/{TASK}/models/{latest_model.name}/")